# Canary Wharf — Fixed-UE Pass SNR (Two GPUs)

Select the **prepared** `original3` or `additional10` campaign in Run Settings using the same
`run_elevation_sim.py` site runner as the other `custom_drivers` notebooks.
The saved configuration, fixed receiver and orbital waypoints are the inputs;
this notebook does not prepare or change them.

Run the cells in order with your Sionna Python kernel on the CUDA host.
Each GPU runs one independent RT sample at a time. The launch cell blocks until
all samples finish, then writes the SNR CSVs and plot. Interrupting that cell
stops its workers; rerun it to skip completed samples and retry unfinished ones.
The ten additional satellites use the same August 2026 historical TLE replay,
fixed receiver, configuration and link budget. Their real predicted peaks and
compass azimuths are varied; orbital elements are not modified. Azimuth is
clockwise from north (0° north, 90° east).


## 1 - Environment Setup

In [1]:
import os
import sys
from pathlib import Path

# Find the repository whether the notebook starts in custom_drivers or the repo root.
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/simulations/run_pass_snr_sweep.py").is_file()), None)
if REPO is None:
    REPO = Path.home() / "Ray_Tracing"
if not (REPO / "src/simulations/run_pass_snr_sweep.py").is_file():
    raise FileNotFoundError("Set REPO to your Ray_Tracing repository")
sys.path.insert(0, str(REPO.parent))
os.environ.setdefault("DRJIT_LIBLLVM_PATH", "/usr/lib/x86_64-linux-gnu/libLLVM-20.so")

print(f"Repository: {REPO}")
print(f"Kernel Python: {sys.executable}")
# GPU libraries are initialized only inside the site-runner subprocesses.

Repository: /path/to/Ray_Tracing
Kernel Python: /path/to/sionna-venv/bin/python


## 2 - Imports

In [2]:
import csv
import json
import subprocess
from collections import Counter
from IPython.display import Image, Markdown, display
import importlib
from Ray_Tracing.src.simulations import run_pass_snr_sweep
# Pick up scheduler fixes when rerunning this cell in an existing kernel.
importlib.reload(run_pass_snr_sweep)
from Ray_Tracing.src.simulations.run_pass_snr_sweep import run_campaign, collect

## 3 - Run Settings

Only execution settings and the analysis threshold are configured here.
Sampling resolution, ray counts and link-budget assumptions come from the
prepared files. Changing `THRESHOLD_DB` requires only reanalysis, not more RT.

In [3]:
# Switch campaigns here; each has its own inputs, samples and analysis outputs.
CAMPAIGNS = {
    "original3": REPO / "Results/wharf_real_pass_snr",
    "additional10": REPO / "Results/wharf_real_pass_snr_additional10",
}
CAMPAIGN = "additional10"
OUTPUT_DIR = CAMPAIGNS[CAMPAIGN]
GPU_IDS = [0, 1]                 # Physical GPUs, one worker per GPU.
PYTHON_EXECUTABLE = sys.executable  # Python environment with patched Sionna + CUDA.
THRESHOLD_DB = 0.0               # Strict SNR > threshold, in dB.

# Prepared NumPy 2 inputs are loaded compatibly by the site runner;
# use the selected Sionna kernel directly, without a compatibility launcher.

## 4 - Inspect Prepared Inputs

In [4]:
manifest = json.loads((OUTPUT_DIR / "campaign.json").read_text())
cfg = json.loads((OUTPUT_DIR / "simulation_config.json").read_text())
for file in (OUTPUT_DIR / "ue.pkl", Path(cfg["SCENE_XML"])):
    if not file.is_file():
        raise FileNotFoundError(file)
with (OUTPUT_DIR / "geometry.csv").open(newline="") as f:
    geometry = list(csv.DictReader(f))
counts = Counter(row["pass_id"] for row in geometry)

print(f"Fixed UE XYZ (m): {manifest['ue_scene_xyz_m']}")
print(f"Carrier: {cfg['FREQUENCY'] / 1e9:g} GHz; TX: {cfg['TX_POWER_DBM']:g} dBm")
print(f"Bandwidth: {manifest['bandwidth_hz'] / 1e6:g} MHz; noise figure: {manifest['noise_figure_db']:g} dB")
print(f"Rays per solve: {cfg['BASE_SAMPLES_PER_SRC']:,}")
print("Saved TLE replay: geometry will be used as prepared.")
rows = ["| Satellite / NORAD | Peak (deg) | Peak azimuth (deg) | Interval (s) | RT solves | Complete |",
        "|---|---:|---:|---:|---:|---:|"]
for meta in manifest["passes"]:
    folder = OUTPUT_DIR / meta["directory"]
    if not (folder / "waypoints.pkl").is_file():
        raise FileNotFoundError(folder / "waypoints.pkl")
    complete = sum(len(list((folder / f"sample_{i:05d}").glob("results_elev*deg.pkl"))) == 1
                   for i in range(counts[folder.name]))
    peak_az = meta.get("peak_azimuth_deg")
    az_text = f"{peak_az:.1f}" if peak_az is not None else "—"
    rows.append(f"| {meta['satellite_name']} / {meta['norad_id']} | {meta['actual_max_elevation_deg']:.2f} | {az_text} | "
                f"{meta['sample_interval_s']:.3f} | {counts[folder.name]} | {complete} |")
display(Markdown("\n".join(rows)))
print(f"Total: {sum(counts.values())} RT solves across both rise/set legs")

Fixed UE XYZ (m): [-4.7447509765625, -6.0, 1.5]
Carrier: 2 GHz; TX: 43 dBm
Bandwidth: 10 MHz; noise figure: 7 dB
Rays per solve: 10,000,000
Saved TLE replay: geometry will be used as prepared.


| Satellite / NORAD | Peak (deg) | Peak azimuth (deg) | Interval (s) | RT solves | Complete |
|---|---:|---:|---:|---:|---:|
| STARLINK-5341 / 55287 | 14.70 | 33.7 | 4.975 | 132 | 0 |
| STARLINK-34478 / 64435 | 24.94 | 316.7 | 4.991 | 144 | 0 |
| STARLINK-35266 / 65804 | 34.88 | 68.2 | 4.987 | 139 | 0 |
| STARLINK-5643 / 55431 | 46.11 | 308.4 | 4.968 | 152 | 0 |
| STARLINK-5972 / 56362 | 54.93 | 100.7 | 4.976 | 131 | 0 |
| STARLINK-37638 / 69144 | 65.65 | 258.2 | 4.989 | 131 | 0 |
| STARLINK-31046 / 58552 | 74.82 | 159.4 | 4.982 | 137 | 0 |
| STARLINK-3629 / 51774 | 85.07 | 197.7 | 4.972 | 141 | 0 |
| STARLINK-34568 / 64717 | 29.51 | 249.3 | 4.981 | 145 | 0 |
| STARLINK-1329 / 45531 | 60.49 | 357.2 | 4.988 | 109 | 0 |

Total: 1361 RT solves across both rise/set legs


## 5 - Check GPUs and Patched Sionna

The child processes receive their own `CUDA_VISIBLE_DEVICES` before GPU
imports. Physical GPU 0 or 1 becomes logical GPU 0 inside its worker.

In [5]:
if not GPU_IDS or len(set(GPU_IDS)) != len(GPU_IDS) or any(g < 0 for g in GPU_IDS):
    raise ValueError("Choose distinct nonnegative physical GPU IDs")
gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.free", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True)
print(gpu_check.stdout)
available = {int(row[0]) for row in csv.reader(gpu_check.stdout.splitlines())}
if not set(GPU_IDS).issubset(available):
    raise ValueError(f"Requested GPUs {GPU_IDS}; available IDs: {sorted(available)}")

patch_check = subprocess.run(
    [str(PYTHON_EXECUTABLE), str(REPO / "src/sionna_patch_check.py")],
    cwd=REPO, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(GPU_IDS[0])),
    capture_output=True, text=True)
print(patch_check.stdout)
if patch_check.returncode:
    print(patch_check.stderr)
    raise RuntimeError("The selected child Python must load the repository's patched Sionna")

0, NVIDIA GeForce RTX 3080, 9873
1, NVIDIA GeForce RTX 3080, 9867

{
  "ok": true,
  "sionna_version": "1.2.1",
  "version_supported": true,
  "beamforming": {
    "ok": true,
    "reason": "PathSolver.__call__ accepts beamforming, beam_angle"
  },
  "epsilon_guard": {
    "ok": true,
    "cap_m": 0.001,
    "reason": "guard capped in spawn_ray_to"
  },
  "manifest": {
    "ok": null,
    "reason": "no manifest at /path/to/Ray_Tracing/src/sionna_patch/MANIFEST.json"
  },
  "archive_rev": "2026-09-24",
  "cuda_available": true
}
sionna 1.2.1: beamforming=yes guard=1.0 mm cuda=yes archive=2026-09-24



## 6 - Launch Simulations (Multi-GPU)

A shared queue feeds both GPUs, with at most one site-runner process per GPU.
Each sample uses its saved waypoint index and its own output directory, so
rising/setting samples with similar elevations do not collide. Logs are in
`<pass>/sample_<index>/run.log`. Failed samples restart with `--no-resume`;
completed samples are retained. A failure stops the other active worker and
reports the failing log path. Analysis runs only after all samples finish.

In [ ]:
run_campaign(
    output=OUTPUT_DIR,
    gpu_ids=GPU_IDS,
    python=PYTHON_EXECUTABLE,
    threshold_db=THRESHOLD_DB,
)

1361 samples: 0 complete, 1361 pending; GPUs (0, 1)
GPU 0, PID 386407: pass_00_norad55287 sample 1/132
GPU 1, PID 386408: pass_00_norad55287 sample 2/132
Completed 1/1361 (0.3 min)
GPU 0, PID 388492: pass_00_norad55287 sample 3/132
Completed 2/1361 (0.4 min)
GPU 1, PID 389627: pass_00_norad55287 sample 4/132
Completed 3/1361 (0.6 min)
GPU 0, PID 391602: pass_00_norad55287 sample 5/132
Completed 4/1361 (0.8 min)
GPU 1, PID 392703: pass_00_norad55287 sample 6/132
Completed 5/1361 (1.0 min)
GPU 0, PID 394612: pass_00_norad55287 sample 7/132
Completed 6/1361 (1.2 min)
GPU 1, PID 395882: pass_00_norad55287 sample 8/132
Completed 7/1361 (1.4 min)
GPU 0, PID 397711: pass_00_norad55287 sample 9/132
Completed 8/1361 (1.5 min)
GPU 1, PID 398956: pass_00_norad55287 sample 10/132
Completed 9/1361 (1.8 min)
GPU 0, PID 400959: pass_00_norad55287 sample 11/132
Completed 10/1361 (1.9 min)
GPU 1, PID 402161: pass_00_norad55287 sample 12/132
Completed 11/1361 (2.2 min)
GPU 0, PID 403923: pass_00_norad55

## 7 - View SNR Results

In [ ]:
display(Image(filename=str(OUTPUT_DIR / "pass_snr.png")))
with (OUTPUT_DIR / "elevation_bins.csv").open(newline="") as f:
    bins = list(csv.DictReader(f))
rows = ["| Pass | Elevation bin (deg) | Samples | SNR > threshold (%) |",
        "|---|---:|---:|---:|"]
for row in bins:
    rows.append(f"| {row['pass_id']} | {row['elevation_lo_deg']}–{row['elevation_hi_deg']} | "
                f"{row['sample_count']} | {float(row['percent_above_threshold']):.1f} |")
display(Markdown("\n".join(rows)))
print(f"Per-sample data: {OUTPUT_DIR / 'snr_samples.csv'}")

## 8 - Optional: Reanalyze with a Different Threshold

Run this cell only when changing the SNR threshold after simulation. Then
rerun the view cell above. Incomplete campaigns are rejected by analysis.

In [ ]:
# THRESHOLD_DB = 3.0
# collect(OUTPUT_DIR, THRESHOLD_DB)